# Publish NB1 TFRecords → Private Kaggle Dataset

**Smart Sniper Spotter · Stage 2 · Transfer infrastructure**

Packages Notebook 1's TFRecord output (~10GB: 16 train + 4 val shards plus
label_map and metadata) as a versioned private Kaggle Dataset that the
Colab training notebooks can fetch.


**Dataset slug:** `giladfaibish/smart-spotter-tfrecords-v1`
- `v1` in the name so we can publish `v2` later if NB1's output changes
  meaningfully (filter logic, merge ratios, augmentation pipeline).
  Versioning by overwriting the same slug is also possible (Kaggle keeps
  version history), but a new slug forces an explicit decision on the Colab
  side about which version to consume.
- Private. Will be redistributable only to your Kaggle account.

**Prereqs:**
- NB1 output attached to this notebook (so `/kaggle/input/<nb1-slug>/` exists)
- `kaggle.json` uploaded as a Kaggle secret OR pasted inline (see Cell #1)

**One-time cost:** ~10–15 min to upload 10GB to Kaggle's dataset storage.


## Cell #1 — Kaggle API credentials


In [1]:
# Cell #1: Configure Kaggle API credentials inside this Kaggle notebook.
#
# Kaggle notebooks already have a kaggle.json mounted, but it's the
# *current notebook session's* token, which has read access. Publishing a
# dataset needs WRITE access, which requires your account's API key.
#
# Two ways to provide it:
#   (a) Add it as a Kaggle Secret named "KAGGLE_USERNAME" + "KAGGLE_KEY"
#       via the Kaggle notebook's "Add-ons → Secrets" menu. Recommended.
#   (b) Paste the contents below temporarily (delete before saving notebook).
#
# Method (a) is much safer — secrets are not stored in the notebook .ipynb.

import os
import json
from pathlib import Path

USE_SECRETS = True  # set False if pasting inline

if USE_SECRETS:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    KAGGLE_USERNAME = secrets.get_secret("KAGGLE_USERNAME")
    KAGGLE_KEY = secrets.get_secret("KAGGLE_KEY")
else:
    # ONLY for one-off testing — delete the values before saving the notebook.
    KAGGLE_USERNAME = "giladfaibish"
    KAGGLE_KEY = "PASTE_KEY_HERE"

# Write kaggle.json where the kaggle CLI expects it.
kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)
(kaggle_dir / "kaggle.json").write_text(
    json.dumps({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}))
(kaggle_dir / "kaggle.json").chmod(0o600)

# Verify CLI sees the credentials.
!kaggle --version
!kaggle datasets list --user {KAGGLE_USERNAME} 2>&1 | head -5

print(f"\n✓ Kaggle CLI authenticated as: {KAGGLE_USERNAME}")


Kaggle CLI 2.0.0
No datasets found

✓ Kaggle CLI authenticated as: giladfaibish


## Cell #2 — Locate NB1 output


In [2]:
# Cell #2: Locate NB1's output on this notebook's filesystem.
#
# When you "Add data → Notebook Output" from this notebook's sidebar, Kaggle
# mounts NB1's /kaggle/working/ at /kaggle/input/<nb1-slug>/. The exact slug
# depends on how NB1 was saved.

import os
from pathlib import Path

# List everything Kaggle mounted under /kaggle/input/
print(">> Mounted datasets under /kaggle/input/:")
!ls -la /kaggle/input/

# Auto-detect NB1's output by looking for the TFRecord shard naming pattern.
# Expected: 16 train shards + 4 val shards under some subdirectory.
import glob
candidates = sorted(glob.glob("/kaggle/input/*/train-*.tfrecord*"))
if not candidates:
    # NB1 might have written to nested directories.
    candidates = sorted(glob.glob("/kaggle/input/**/train-*.tfrecord*", recursive=True))

assert candidates, (
    "No train-*.tfrecord files found under /kaggle/input/. "
    "Did you attach NB1's output? Sidebar → Add data → Notebook Output → select NB1.")

# All train shards should be in the same directory.
NB1_TFRECORD_DIR = os.path.dirname(candidates[0])
print(f"\n>> Detected NB1 TFRecord directory: {NB1_TFRECORD_DIR}")

train_shards = sorted(glob.glob(os.path.join(NB1_TFRECORD_DIR, "train-*.tfrecord*")))
val_shards = sorted(glob.glob(os.path.join(NB1_TFRECORD_DIR, "val-*.tfrecord*")))
print(f"   Train shards: {len(train_shards)}")
print(f"   Val shards:   {len(val_shards)}")

assert len(train_shards) == 16, f"Expected 16 train shards from NB1, got {len(train_shards)}"
assert len(val_shards) == 4, f"Expected 4 val shards from NB1, got {len(val_shards)}"

# Total size estimate.
total_bytes = sum(os.path.getsize(p) for p in train_shards + val_shards)
print(f"   Total TFRecord size: {total_bytes/1e9:.2f} GB")

# Look for label_map.pbtxt and any metadata file NB1 wrote.
nb1_root = os.path.dirname(NB1_TFRECORD_DIR) if NB1_TFRECORD_DIR.endswith("tfrecords") else NB1_TFRECORD_DIR
NB1_LABEL_MAP = None
for candidate in ["label_map.pbtxt", "../label_map.pbtxt"]:
    p = os.path.join(NB1_TFRECORD_DIR, candidate)
    if os.path.exists(p):
        NB1_LABEL_MAP = os.path.abspath(p)
        break
if NB1_LABEL_MAP is None:
    # Search wider.
    hits = glob.glob("/kaggle/input/**/label_map.pbtxt", recursive=True)
    if hits:
        NB1_LABEL_MAP = hits[0]
assert NB1_LABEL_MAP, "label_map.pbtxt not found in NB1 output."
print(f"   Label map: {NB1_LABEL_MAP}")

# Metadata JSON (optional but useful — NB1 should have written dataset stats).
NB1_METADATA = None
hits = glob.glob("/kaggle/input/**/dataset_metadata*.json", recursive=True)
if hits:
    NB1_METADATA = hits[0]
    print(f"   Metadata:  {NB1_METADATA}")
else:
    print(f"   Metadata:  (none found — will skip)")


>> Mounted datasets under /kaggle/input/:
total 12
drwxr-xr-x 3 root root 4096 May 18 17:13 .
drwxr-xr-x 5 root root 4096 May 18 17:13 ..
drwxr-xr-x 3 root root 4096 May 18 17:13 notebooks

>> Detected NB1 TFRecord directory: /kaggle/input/notebooks/giladfaibish/data-preparation/snipeit_person_dataset/tfrecords
   Train shards: 16
   Val shards:   4
   Total TFRecord size: 11.30 GB
   Label map: /kaggle/input/notebooks/giladfaibish/data-preparation/snipeit_person_dataset/label_map.pbtxt
   Metadata:  /kaggle/input/notebooks/giladfaibish/data-preparation/snipeit_person_dataset/dataset_metadata.json


## Cell #3 — Stage to working dir and validate


In [3]:
# Cell #3: Stage everything in a clean working directory and validate.
#
# Kaggle's `kaggle datasets create/version` uploads files but the v2.x CLI
# silently SKIPS subdirectories unless --dir-mode is passed. To avoid this
# trap, we stage all shards FLAT at the top level of the upload dir alongside
# label_map.pbtxt and the metadata files — no subdirectories.
#
# We also run a quick TFRecord read-back to confirm nothing is corrupt
# before the slow upload.

import os
import shutil
import tensorflow as tf

STAGING = "/kaggle/working/dataset_staging"
if os.path.exists(STAGING):
    shutil.rmtree(STAGING)
os.makedirs(STAGING)

print(">> Copying TFRecord shards to staging dir (flat layout)...")
for src in train_shards + val_shards:
    dst = os.path.join(STAGING, os.path.basename(src))
    shutil.copy2(src, dst)
print(f"   Copied {len(train_shards) + len(val_shards)} shards.")

shutil.copy2(NB1_LABEL_MAP, os.path.join(STAGING, "label_map.pbtxt"))
print(f"   Copied label_map.pbtxt")

if NB1_METADATA:
    shutil.copy2(NB1_METADATA, os.path.join(STAGING, "dataset_metadata.json"))
    print(f"   Copied dataset_metadata.json")

# Read-back validation: parse the first example from each split to confirm
# TFRecords are intact post-copy.
print("\n>> Validating TFRecords by reading first example from each split...")

def read_first(pattern):
    paths = sorted(__import__('glob').glob(pattern))
    for raw in tf.data.TFRecordDataset(paths).take(1):
        ex = tf.train.Example()
        ex.ParseFromString(raw.numpy())
        feat = ex.features.feature
        return {
            'filename': feat['image/filename'].bytes_list.value[0].decode(),
            'width': feat['image/width'].int64_list.value[0],
            'height': feat['image/height'].int64_list.value[0],
            'n_boxes': len(feat['image/object/bbox/xmin'].float_list.value),
        }

train_sample = read_first(os.path.join(STAGING, "train-*.tfrecord*"))
val_sample = read_first(os.path.join(STAGING, "val-*.tfrecord*"))
print(f"   Train first example: {train_sample}")
print(f"   Val first example:   {val_sample}")
assert train_sample['n_boxes'] > 0, "Train example has 0 boxes — bad shard."
assert val_sample['n_boxes'] > 0, "Val example has 0 boxes — bad shard."

# Final staging-dir listing.
print(f"\n>> Staging dir size:")
!du -sh {STAGING}
print(f"\n>> Staging dir contents (top of listing):")
!ls -la {STAGING} | head -30

2026-05-18 17:18:03.406018: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779124683.814335      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779124683.916483      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779124684.712814      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779124684.712881      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779124684.712883      57 computation_placer.cc:177] computation placer alr

>> Copying TFRecord shards to staging dir...
   Copied 20 shards.
   Copied label_map.pbtxt
   Copied dataset_metadata.json

>> Validating TFRecords by reading first example from each split...


2026-05-18 17:22:20.402109: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


   Train first example: {'filename': '000000135749.jpg', 'width': 375, 'height': 500, 'n_boxes': 9}
   Val first example:   {'filename': '000000279073.jpg', 'width': 640, 'height': 425, 'n_boxes': 4}

>> Staging dir size:
11G	/kaggle/working/dataset_staging

>> Staging dir contents:
total 20
drwxr-xr-x 3 root root 4096 May 18 17:22 .
drwxr-xr-x 4 root root 4096 May 18 17:18 ..
-rw-r--r-- 1 root root  775 May 18 17:11 dataset_metadata.json
-rw-r--r-- 1 root root   34 May 18 17:11 label_map.pbtxt
drwxr-xr-x 2 root root 4096 May 18 17:22 tfrecords
train-00000-of-00016.tfrecord
train-00001-of-00016.tfrecord
train-00002-of-00016.tfrecord
train-00003-of-00016.tfrecord
train-00004-of-00016.tfrecord
train-00005-of-00016.tfrecord
train-00006-of-00016.tfrecord
train-00007-of-00016.tfrecord
train-00008-of-00016.tfrecord
train-00009-of-00016.tfrecord
train-00010-of-00016.tfrecord
train-00011-of-00016.tfrecord
train-00012-of-00016.tfrecord
train-00013-of-00016.tfrecord
train-00014-of-00016.tfrecord

## Cell #4 — Write Kaggle dataset metadata


In [4]:
# Cell #4: Write the dataset metadata JSON that Kaggle requires.
#
# `kaggle datasets create` reads dataset-metadata.json from the upload dir
# to know the slug, title, license, etc. We write it programmatically so
# the slug, version, and description are sourced from variables we can edit.

import json

DATASET_SLUG = "giladfaibish/smart-spotter-tfrecords-v1"
DATASET_TITLE = "Smart Sniper Spotter — Stage 2 TFRecords v1"
DATASET_DESCRIPTION = """Pre-processed TFRecords for the Smart Sniper Spotter
Target Detection module. Merged COCO 2017 outdoor-person subset + WiderPerson,
~65K train / ~7.3K val examples, sharded for the TF2 Object Detection API.

Contents (all files at top level — no subdirectories):
- train-00000-of-00016.tfrecord ... train-00015-of-00016.tfrecord
- val-00000-of-00004.tfrecord ... val-00003-of-00004.tfrecord
- label_map.pbtxt (1-class: person)
- dataset_metadata.json (split sizes, source counts)

Single class: person.
Source: merged COCO 2017 (awsaf49 subset) + WiderPerson (ritaj1/widerperson).
"""

metadata = {
    "title": DATASET_TITLE,
    "id": DATASET_SLUG,
    "licenses": [{"name": "other"}],  # Kaggle requires this field
    "subtitle": "Pre-processed TFRecords for Stage 2 training",
    "description": DATASET_DESCRIPTION,
    "isPrivate": True,
}

md_path = os.path.join(STAGING, "dataset-metadata.json")
with open(md_path, "w") as f:
    json.dump(metadata, f, indent=2)

print(f">> Wrote dataset-metadata.json")
print(json.dumps(metadata, indent=2))


>> Wrote dataset-metadata.json
{
  "title": "Smart Sniper Spotter \u2014 Stage 2 TFRecords v1",
  "id": "giladfaibish/smart-spotter-tfrecords-v1",
  "licenses": [
    {
      "name": "other"
    }
  ],
  "subtitle": "Pre-processed TFRecords for Stage 2 training",
  "description": "Pre-processed TFRecords for the Smart Sniper Spotter\nTarget Detection module. Merged COCO 2017 outdoor-person subset + WiderPerson,\n~65K train / ~7.3K val examples, sharded for the TF2 Object Detection API.\n\nContents:\n- tfrecords/train-00000-of-00016.tfrecord ... train-00015-of-00016.tfrecord\n- tfrecords/val-00000-of-00004.tfrecord ... val-00003-of-00004.tfrecord\n- label_map.pbtxt (1-class: person)\n- dataset_metadata.json (split sizes, source counts)\n\nSingle class: person.\nSource: merged COCO 2017 (awsaf49 subset) + WiderPerson (ritaj1/widerperson).\n",
  "isPrivate": true
}


## Cell #5 — Upload to Kaggle


In [5]:
# Cell #5: Upload the staging dir as a new version of the private Kaggle Dataset.
#
# On first run, the dataset slug doesn't exist and we create it.
# On subsequent runs, we upload as a new version.
#
# We try version first (since the slug exists for any re-run) and fall back
# to create. This is the inverse of the previous logic, which assumed first-run
# was the common case — it isn't, once the dataset is established.

import subprocess

CREATE_CMD = ["kaggle", "datasets", "create", "-p", STAGING]
VERSION_CMD = ["kaggle", "datasets", "version", "-p", STAGING,
               "-m", "Re-upload from NB1 output"]

print(">> Attempting `kaggle datasets version` (works if slug exists)...")
result = subprocess.run(VERSION_CMD, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    err = (result.stderr or "") + result.stdout
    if "does not exist" in err.lower() or "not found" in err.lower():
        print("\n>> Dataset slug doesn't exist yet — creating it...")
        result = subprocess.run(CREATE_CMD, capture_output=True, text=True)
        print(result.stdout)
        if result.returncode != 0:
            print("Create stderr:", result.stderr)
            raise RuntimeError("Both version and create failed. See errors above.")
    else:
        print("Version stderr:", result.stderr)
        raise RuntimeError(f"Dataset upload failed: {result.stderr}")

print(f"\n✓ Upload complete.")
print(f"   Slug:    {DATASET_SLUG}")
print(f"   URL:     https://www.kaggle.com/datasets/{DATASET_SLUG}")
print(f"\n   In Colab, fetch with:")
print(f"     kaggle datasets download -d {DATASET_SLUG}")
print(f"\n   First Colab download may take ~2 min; subsequent fetches with")
print(f"   the same version are cached server-side.")

>> Attempting `kaggle datasets create`...
Starting upload for file dataset_metadata.json
Upload successful: dataset_metadata.json (775B)
Skipping folder: tfrecords; use '--dir-mode' to upload folders
Starting upload for file label_map.pbtxt
Upload successful: label_map.pbtxt (34B)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/giladfaibish/smart-spotter-tfrecords-v1


✓ Upload complete.
   Slug:    giladfaibish/smart-spotter-tfrecords-v1
   URL:     https://www.kaggle.com/datasets/giladfaibish/smart-spotter-tfrecords-v1

   In Colab, fetch with:
     kaggle datasets download -d giladfaibish/smart-spotter-tfrecords-v1

   First Colab download may take ~2 min; subsequent fetches with
   the same version are cached server-side.
